# Business Operations Analytics Database Setup

## Purpose

This notebook creates a reproducible MySQL database environment for the **Business Operations SQL Analytics** portfolio project.

It will:

1. Connect securely to MySQL
2. Create a separate demonstration database
3. Create the relational table structure
4. Load fictional sample business data
5. Validate record counts and relationships

### Demo Database

`business_operations_analytics_demo`

This database is intentionally separate from the original working database so the existing project environment remains unchanged.

All sample customers, employees, products, transactions, and financial records in this project are fictional and created solely for portfolio and educational purposes.

In [14]:
import mysql.connector
from getpass import getpass

db_password = getpass("Enter MySQL root password: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password
)

cursor = conn.cursor()

print("Connected successfully to MySQL.")

Enter MySQL root password:  ········


Connected successfully to MySQL.


## 1. Create a Fresh Demo Database

This step creates a dedicated MySQL database for the public portfolio project.

The setup notebook intentionally rebuilds only:

`business_operations_analytics_demo`

each time it is run.

This makes the project reproducible and prevents duplicate sample records while leaving the original working database, `business_operations_analytics`, completely unchanged.

In [15]:
demo_database = "business_operations_analytics_demo"

cursor.execute(f"DROP DATABASE IF EXISTS {demo_database}")
cursor.execute(f"CREATE DATABASE {demo_database}")
cursor.execute(f"USE {demo_database}")

print(f"Fresh demo database created: {demo_database}")

Fresh demo database created: business_operations_analytics_demo


## 2. Create Relational Database Schema

This section creates the eight core tables used in the Business Operations Analytics project:

- departments
- employees
- customers
- products
- orders
- order_items
- payments
- expenses

The schema includes primary keys, foreign keys, unique constraints, timestamps, and financial data types to model a realistic small-business operations environment.

In [16]:
schema_sql = """
CREATE TABLE departments (
    department_id INT AUTO_INCREMENT PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL UNIQUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE employees (
    employee_id INT AUTO_INCREMENT PRIMARY KEY,
    department_id INT,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    job_title VARCHAR(100),
    email VARCHAR(120) UNIQUE,
    hire_date DATE,
    salary DECIMAL(12,2),
    employment_status VARCHAR(30) DEFAULT 'Active',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_employee_department
        FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);

CREATE TABLE customers (
    customer_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_name VARCHAR(150) NOT NULL,
    customer_type VARCHAR(50),
    email VARCHAR(120),
    phone VARCHAR(30),
    city VARCHAR(80),
    province_state VARCHAR(80),
    country VARCHAR(80) DEFAULT 'Canada',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE products (
    product_id INT AUTO_INCREMENT PRIMARY KEY,
    product_name VARCHAR(150) NOT NULL,
    category VARCHAR(100),
    unit_price DECIMAL(10,2) NOT NULL,
    active_status VARCHAR(20) DEFAULT 'Active',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE orders (
    order_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    employee_id INT,
    order_date DATE NOT NULL,
    order_status VARCHAR(30) DEFAULT 'Pending',
    total_amount DECIMAL(12,2) DEFAULT 0.00,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_order_customer
        FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id),
    CONSTRAINT fk_order_employee
        FOREIGN KEY (employee_id)
        REFERENCES employees(employee_id)
);

CREATE TABLE order_items (
    order_item_id INT AUTO_INCREMENT PRIMARY KEY,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL,
    line_total DECIMAL(12,2) NOT NULL,
    CONSTRAINT fk_orderitem_order
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id),
    CONSTRAINT fk_orderitem_product
        FOREIGN KEY (product_id)
        REFERENCES products(product_id)
);

CREATE TABLE payments (
    payment_id INT AUTO_INCREMENT PRIMARY KEY,
    order_id INT NOT NULL,
    payment_date DATE NOT NULL,
    payment_method VARCHAR(50),
    amount DECIMAL(12,2) NOT NULL,
    payment_status VARCHAR(30) DEFAULT 'Completed',
    CONSTRAINT fk_payment_order
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
);

CREATE TABLE expenses (
    expense_id INT AUTO_INCREMENT PRIMARY KEY,
    department_id INT,
    expense_date DATE NOT NULL,
    expense_category VARCHAR(100),
    description VARCHAR(255),
    amount DECIMAL(12,2) NOT NULL,
    CONSTRAINT fk_expense_department
        FOREIGN KEY (department_id)
        REFERENCES departments(department_id)
);
"""

for statement in schema_sql.split(";"):
    statement = statement.strip()
    if statement:
        cursor.execute(statement)

conn.commit()

print("Relational schema created successfully.")

Relational schema created successfully.


### Schema Validation

The following check confirms that all required relational tables were created successfully before sample data is loaded.

In [17]:
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()

print("Tables created:")
print("-" * 30)

for table in tables:
    print(table[0])

print("-" * 30)
print(f"Total tables: {len(tables)}")

Tables created:
------------------------------
customers
departments
employees
expenses
order_items
orders
payments
products
------------------------------
Total tables: 8


## 3. Load Departments and Employees

This section loads fictional organizational data used by the portfolio project.

The dataset includes five departments and six employees representing sales, operations, finance, marketing, and information-technology functions.

All names, email addresses, salaries, and employment details are synthetic and created solely for demonstration purposes.

In [18]:
departments_data = [
    ("Sales",),
    ("Operations",),
    ("Finance",),
    ("Marketing",),
    ("Information Technology",)
]

cursor.executemany(
    """
    INSERT INTO departments (department_name)
    VALUES (%s)
    """,
    departments_data
)

employees_data = [
    (1, "David", "Chen", "Sales Manager",
     "david.chen@demo-company.com", "2021-03-15", 78000.00),

    (1, "Sarah", "Patel", "Sales Representative",
     "sarah.patel@demo-company.com", "2022-06-01", 58000.00),

    (2, "Michael", "Brown", "Operations Manager",
     "michael.brown@demo-company.com", "2020-09-10", 82000.00),

    (3, "Emily", "Wilson", "Financial Analyst",
     "emily.wilson@demo-company.com", "2023-01-16", 65000.00),

    (4, "Daniel", "Lee", "Marketing Specialist",
     "daniel.lee@demo-company.com", "2022-11-07", 61000.00),

    (5, "Priya", "Sharma", "Database Analyst",
     "priya.sharma@demo-company.com", "2024-02-12", 72000.00)
]

cursor.executemany(
    """
    INSERT INTO employees
    (
        department_id,
        first_name,
        last_name,
        job_title,
        email,
        hire_date,
        salary
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    employees_data
)

conn.commit()

print("Departments and employees loaded successfully.")

Departments and employees loaded successfully.


## 4. Load Customers and Products / Services

This section loads fictional customer accounts and the products/services offered by the demonstration business.

The customer portfolio is intentionally diversified across multiple business types and locations, while the product catalog represents common technology, database, analytics, and business-intelligence services.

All records are synthetic and created exclusively for portfolio demonstration purposes.

In [19]:
customers_data = [
    ("Maple Tech Solutions", "Business", "contact@mapletech.example", "416-555-0101", "Toronto", "Ontario", "Canada"),
    ("Northern Retail Group", "Business", "contact@northernretail.example", "905-555-0102", "Markham", "Ontario", "Canada"),
    ("Lakeview Services", "Business", "contact@lakeview.example", "647-555-0103", "Mississauga", "Ontario", "Canada"),
    ("Capital Office Group", "Business", "contact@capitaloffice.example", "613-555-0104", "Ottawa", "Ontario", "Canada"),
    ("Metro Business Systems", "Business", "contact@metrobusiness.example", "416-555-0105", "Toronto", "Ontario", "Canada"),
    ("Greenfield Consulting", "Business", "contact@greenfield.example", "905-555-0106", "Vaughan", "Ontario", "Canada"),
    ("BlueSky Logistics", "Business", "contact@bluesky.example", "647-555-0107", "Brampton", "Ontario", "Canada"),
    ("Summit Professional Services", "Business", "contact@summit.example", "905-555-0108", "Richmond Hill", "Ontario", "Canada"),
    ("Urban Data Systems", "Business", "contact@urbandata.example", "416-555-0109", "Toronto", "Ontario", "Canada"),
    ("Evergreen Enterprises", "Business", "contact@evergreen.example", "289-555-0110", "Hamilton", "Ontario", "Canada")
]

cursor.executemany(
    """
    INSERT INTO customers
    (
        customer_name,
        customer_type,
        email,
        phone,
        city,
        province_state,
        country
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,
    customers_data
)


products_data = [
    ("Database Design Service", "Database", 1600.00),
    ("Business Analytics Package", "Analytics", 2500.00),
    ("Power BI Dashboard", "Business Intelligence", 2200.00),
    ("SQL Development Service", "Database", 1800.00),
    ("Data Cleaning Service", "Data Management", 900.00),
    ("Database Optimization", "Database", 1500.00),
    ("Analytics Consultation", "Consulting", 750.00),
    ("Reporting Automation", "Automation", 1200.00),
    ("Data Migration Service", "Data Management", 2000.00),
    ("Performance Analysis", "Analytics", 1100.00)
]

cursor.executemany(
    """
    INSERT INTO products
    (
        product_name,
        category,
        unit_price
    )
    VALUES (%s, %s, %s)
    """,
    products_data
)

conn.commit()

print("Customers and products/services loaded successfully.")

Customers and products/services loaded successfully.


In [20]:
cursor.execute("SELECT COUNT(*) FROM customers")
customer_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM products")
product_count = cursor.fetchone()[0]

print(f"Customers:         {customer_count}")
print(f"Products/Services: {product_count}")

Customers:         10
Products/Services: 10


## 5. Load Orders and Order Items

This section creates the transactional sales data used by the analytics project.

The demonstration dataset contains:

- 20 customer orders
- Transactions spanning January 2025 through August 2026
- Sales activity attributed to two sales employees
- 38 individual order-item records

Order totals are calculated directly from the associated order items rather than entered manually. This helps preserve transactional consistency between detailed sales records and reported revenue.

In [21]:
orders_data = [
    (1, 2, "2025-01-15", "Completed"),
    (2, 1, "2025-02-18", "Completed"),
    (3, 2, "2025-03-12", "Completed"),
    (4, 1, "2025-04-21", "Completed"),
    (5, 2, "2025-05-16", "Completed"),
    (6, 1, "2025-06-20", "Completed"),
    (7, 2, "2025-07-14", "Completed"),
    (8, 1, "2025-08-19", "Completed"),
    (9, 2, "2025-09-17", "Completed"),
    (10, 1, "2025-10-22", "Completed"),

    (1, 2, "2025-11-18", "Completed"),
    (2, 1, "2025-12-16", "Completed"),
    (3, 2, "2026-01-20", "Completed"),
    (4, 1, "2026-02-17", "Completed"),
    (5, 2, "2026-03-19", "Completed"),
    (6, 1, "2026-04-21", "Completed"),
    (7, 2, "2026-05-15", "Completed"),
    (8, 1, "2026-06-18", "Completed"),
    (9, 2, "2026-07-21", "Completed"),
    (10, 1, "2026-08-18", "Completed")
]

cursor.executemany(
    """
    INSERT INTO orders
    (
        customer_id,
        employee_id,
        order_date,
        order_status
    )
    VALUES (%s, %s, %s, %s)
    """,
    orders_data
)

conn.commit()

print("20 orders loaded successfully.")

20 orders loaded successfully.


### Order Item Details

Each order contains one or more products/services.

The `line_total` field represents:

**Quantity × Unit Price**

After the order items are loaded, the notebook calculates each order's total value from its underlying transaction lines.

In [22]:
original_conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=db_password,
    database="business_operations_analytics"
)

original_cursor = original_conn.cursor()

original_cursor.execute("""
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    line_total
FROM order_items
ORDER BY order_item_id
""")

original_order_items = original_cursor.fetchall()

print(f"Order items retrieved: {len(original_order_items)}")

for row in original_order_items:
    print(row)

Order items retrieved: 38
(1, 1, 1, Decimal('2500.00'), Decimal('2500.00'))
(1, 5, 2, Decimal('950.00'), Decimal('1900.00'))
(2, 2, 1, Decimal('3200.00'), Decimal('3200.00'))
(2, 9, 2, Decimal('450.00'), Decimal('900.00'))
(3, 4, 1, Decimal('2200.00'), Decimal('2200.00'))
(3, 7, 1, Decimal('1600.00'), Decimal('1600.00'))
(4, 3, 1, Decimal('1800.00'), Decimal('1800.00'))
(4, 6, 3, Decimal('750.00'), Decimal('2250.00'))
(5, 8, 1, Decimal('2800.00'), Decimal('2800.00'))
(5, 10, 1, Decimal('1200.00'), Decimal('1200.00'))
(6, 5, 3, Decimal('950.00'), Decimal('2850.00'))
(6, 9, 2, Decimal('450.00'), Decimal('900.00'))
(7, 1, 1, Decimal('2500.00'), Decimal('2500.00'))
(7, 4, 1, Decimal('2200.00'), Decimal('2200.00'))
(8, 2, 1, Decimal('3200.00'), Decimal('3200.00'))
(8, 6, 2, Decimal('750.00'), Decimal('1500.00'))
(9, 7, 2, Decimal('1600.00'), Decimal('3200.00'))
(9, 9, 1, Decimal('450.00'), Decimal('450.00'))
(10, 3, 2, Decimal('1800.00'), Decimal('3600.00'))
(10, 10, 1, Decimal('1200.00'), 

In [23]:
customers_data = [
    ("NorthStar Retail Ltd", "Business", "contact@northstar-demo.com", "416-555-0101", "Toronto", "Ontario", "Canada"),
    ("Maple Tech Solutions", "Business", "info@mapletech-demo.com", "905-555-0102", "Markham", "Ontario", "Canada"),
    ("GreenLeaf Foods Inc", "Business", "orders@greenleaf-demo.com", "647-555-0103", "Mississauga", "Ontario", "Canada"),
    ("BlueWave Consulting", "Business", "admin@bluewave-demo.com", "416-555-0104", "Toronto", "Ontario", "Canada"),
    ("Northern Logistics", "Business", "service@northernlogistics-demo.com", "905-555-0105", "Brampton", "Ontario", "Canada"),
    ("Metro Learning Centre", "Business", "office@metrolearning-demo.com", "289-555-0106", "Hamilton", "Ontario", "Canada"),
    ("Capital Office Group", "Business", "purchasing@capitaloffice-demo.com", "613-555-0107", "Ottawa", "Ontario", "Canada"),
    ("Lakeview Services", "Business", "contact@lakeview-demo.com", "705-555-0108", "Barrie", "Ontario", "Canada"),
    ("WestEnd Digital", "Business", "hello@westend-demo.com", "416-555-0109", "Toronto", "Ontario", "Canada"),
    ("Durham Business Solutions", "Business", "info@durhambusiness-demo.com", "905-555-0110", "Oshawa", "Ontario", "Canada")
]

products_data = [
    ("Business Analytics Package", "Analytics", 2500.00),
    ("Database Design Service", "Database", 3200.00),
    ("SQL Performance Audit", "Database", 1800.00),
    ("Power BI Dashboard", "Business Intelligence", 2200.00),
    ("Data Cleaning Service", "Analytics", 950.00),
    ("Monthly Database Support", "Support", 750.00),
    ("Reporting Automation", "Automation", 1600.00),
    ("Data Migration Service", "Database", 2800.00),
    ("Analytics Consultation", "Consulting", 450.00),
    ("SQL Training Workshop", "Training", 1200.00)
]

In [24]:
checks = {}

for table in [
    "departments",
    "employees",
    "customers",
    "products",
    "orders"
]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    checks[table] = cursor.fetchone()[0]

checks

{'departments': 5,
 'employees': 6,
 'customers': 10,
 'products': 10,
 'orders': 20}

## 6. Load Order Item Transactions

This section loads the detailed transaction lines associated with the 20 customer orders.

The demonstration dataset contains **38 order-item records**. Each record identifies:

- Order
- Product / service
- Quantity
- Unit price
- Calculated line total

The values reproduce the validated dataset used by the Business Operations Analytics notebook.

In [25]:
order_items_data = [
    (1, 1, 1, 2500.00, 2500.00),
    (1, 5, 2, 950.00, 1900.00),

    (2, 2, 1, 3200.00, 3200.00),
    (2, 9, 2, 450.00, 900.00),

    (3, 4, 1, 2200.00, 2200.00),
    (3, 7, 1, 1600.00, 1600.00),

    (4, 3, 1, 1800.00, 1800.00),
    (4, 6, 3, 750.00, 2250.00),

    (5, 8, 1, 2800.00, 2800.00),
    (5, 10, 1, 1200.00, 1200.00),

    (6, 5, 3, 950.00, 2850.00),
    (6, 9, 2, 450.00, 900.00),

    (7, 1, 1, 2500.00, 2500.00),
    (7, 4, 1, 2200.00, 2200.00),

    (8, 2, 1, 3200.00, 3200.00),
    (8, 6, 2, 750.00, 1500.00),

    (9, 7, 2, 1600.00, 3200.00),
    (9, 9, 1, 450.00, 450.00),

    (10, 3, 2, 1800.00, 3600.00),
    (10, 10, 1, 1200.00, 1200.00),

    (11, 1, 1, 2500.00, 2500.00),
    (11, 6, 2, 750.00, 1500.00),

    (12, 4, 2, 2200.00, 4400.00),

    (13, 2, 1, 3200.00, 3200.00),
    (13, 5, 1, 950.00, 950.00),

    (14, 8, 1, 2800.00, 2800.00),
    (14, 9, 3, 450.00, 1350.00),

    (15, 7, 1, 1600.00, 1600.00),
    (15, 10, 2, 1200.00, 2400.00),

    (16, 3, 1, 1800.00, 1800.00),
    (16, 6, 3, 750.00, 2250.00),

    (17, 1, 2, 2500.00, 5000.00),

    (18, 4, 1, 2200.00, 2200.00),
    (18, 5, 2, 950.00, 1900.00),

    (19, 2, 1, 3200.00, 3200.00),
    (19, 7, 1, 1600.00, 1600.00),

    (20, 8, 1, 2800.00, 2800.00),
    (20, 9, 2, 450.00, 900.00)
]

cursor.executemany(
    """
    INSERT INTO order_items
    (
        order_id,
        product_id,
        quantity,
        unit_price,
        line_total
    )
    VALUES (%s, %s, %s, %s, %s)
    """,
    order_items_data
)

conn.commit()

print(f"Order items loaded successfully: {len(order_items_data)}")

Order items loaded successfully: 38


### Calculate Order Totals

Instead of manually entering order totals, each order value is calculated from its detailed order-item records.

This provides a database-level consistency check between transaction details and reported revenue.

In [26]:
cursor.execute("""
UPDATE orders o
JOIN (
    SELECT
        order_id,
        SUM(line_total) AS calculated_total
    FROM order_items
    GROUP BY order_id
) totals
    ON o.order_id = totals.order_id
SET o.total_amount = totals.calculated_total
""")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM order_items")
order_item_count = cursor.fetchone()[0]

cursor.execute("SELECT SUM(total_amount) FROM orders")
total_revenue = cursor.fetchone()[0]

print(f"Order items:   {order_item_count}")
print(f"Total revenue: ${total_revenue:,.2f}")

Order items:   38
Total revenue: $84,300.00


## 7. Load Payments

This section creates one completed payment record for each completed order.

Payment values are derived from each order's calculated total so that reported revenue and received payments reconcile exactly.

In [27]:
cursor.execute("""
INSERT INTO payments
(
    order_id,
    payment_date,
    payment_method,
    amount,
    payment_status
)
SELECT
    order_id,
    order_date,
    CASE
        WHEN MOD(order_id, 4) = 0 THEN 'Credit Card'
        WHEN MOD(order_id, 4) = 1 THEN 'Bank Transfer'
        WHEN MOD(order_id, 4) = 2 THEN 'E-Transfer'
        ELSE 'Debit Card'
    END,
    total_amount,
    'Completed'
FROM orders
""")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM payments")
payment_count = cursor.fetchone()[0]

cursor.execute("SELECT SUM(amount) FROM payments")
payments_received = cursor.fetchone()[0]

print(f"Payments:          {payment_count}")
print(f"Payments received: ${payments_received:,.2f}")

Payments:          20
Payments received: $84,300.00


## 8. Load Operating Expenses

This section loads 20 fictional operating-expense records across sales, operations, finance, marketing, and information-technology departments.

The expense data supports profitability, margin, and monthly financial-performance analysis.

In [28]:
expenses_data = [
    (1, "2025-01-10", "Travel", "Client meetings and local travel", 850.00),
    (2, "2025-02-05", "Software", "Operations software subscription", 1200.00),
    (3, "2025-03-01", "Accounting", "Accounting and compliance services", 950.00),
    (4, "2025-04-12", "Advertising", "Digital marketing campaign", 1800.00),
    (5, "2025-05-08", "Cloud Services", "Database and cloud hosting", 1450.00),
    (1, "2025-06-14", "Training", "Sales training workshop", 700.00),
    (2, "2025-07-03", "Equipment", "Office and operations equipment", 2100.00),
    (3, "2025-08-11", "Professional Services", "Financial consulting", 1250.00),
    (4, "2025-09-16", "Advertising", "Social media campaign", 1600.00),
    (5, "2025-10-09", "Software", "Database tools and licenses", 1750.00),
    (2, "2025-11-21", "Office", "Office supplies and maintenance", 900.00),
    (3, "2025-12-15", "Accounting", "Year-end accounting services", 1400.00),

    (1, "2026-01-09", "Travel", "Client meetings and local travel", 900.00),
    (2, "2026-02-06", "Software", "Operations platform subscription", 1250.00),
    (3, "2026-03-03", "Accounting", "Financial reporting services", 1000.00),
    (4, "2026-04-10", "Advertising", "Digital advertising campaign", 1900.00),
    (5, "2026-05-07", "Cloud Services", "Cloud database infrastructure", 1550.00),
    (1, "2026-06-13", "Training", "Sales development program", 750.00),
    (2, "2026-07-04", "Equipment", "Operations equipment upgrade", 2200.00),
    (5, "2026-08-01", "Software", "Database and analytics tools", 1800.00)
]

cursor.executemany(
    """
    INSERT INTO expenses
    (
        department_id,
        expense_date,
        expense_category,
        description,
        amount
    )
    VALUES (%s, %s, %s, %s, %s)
    """,
    expenses_data
)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM expenses")
expense_count = cursor.fetchone()[0]

cursor.execute("SELECT SUM(amount) FROM expenses")
total_expenses = cursor.fetchone()[0]

print(f"Expenses:       {expense_count}")
print(f"Total expenses: ${total_expenses:,.2f}")

Expenses:       20
Total expenses: $27,300.00


## 9. Final Database Validation

This final validation confirms that the demonstration database reproduces the complete dataset used by the Business Operations SQL Analytics portfolio notebook.

Expected record counts:

- 5 departments
- 6 employees
- 10 customers
- 10 products/services
- 20 orders
- 38 order items
- 20 payments
- 20 expenses

Expected financial totals:

- Total Revenue: $84,300
- Payments Received: $84,300
- Total Expenses: $27,300
- Operating Surplus: $57,000

In [29]:
validation = {}

for table in [
    "departments",
    "employees",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "expenses"
]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    validation[table] = cursor.fetchone()[0]

cursor.execute("SELECT SUM(total_amount) FROM orders")
revenue = cursor.fetchone()[0]

cursor.execute("SELECT SUM(amount) FROM payments")
payments = cursor.fetchone()[0]

cursor.execute("SELECT SUM(amount) FROM expenses")
expenses = cursor.fetchone()[0]

operating_surplus = revenue - expenses

print("=" * 60)
print("          FINAL DATABASE VALIDATION")
print("=" * 60)

for table, count in validation.items():
    print(f"{table:<15} {count}")

print("-" * 60)
print(f"Total Revenue:       ${revenue:,.2f}")
print(f"Payments Received:   ${payments:,.2f}")
print(f"Total Expenses:      ${expenses:,.2f}")
print(f"Operating Surplus:   ${operating_surplus:,.2f}")
print("=" * 60)

          FINAL DATABASE VALIDATION
departments     5
employees       6
customers       10
products        10
orders          20
order_items     38
payments        20
expenses        20
------------------------------------------------------------
Total Revenue:       $84,300.00
Payments Received:   $84,300.00
Total Expenses:      $27,300.00
Operating Surplus:   $57,000.00
